In [ ]:
# Source Code 10 (ArcGIS Pro environment)
# Script to convert GeoTIFF files into shapefiles in ArcGIS Pro environment.

import arcpy  # Provides access to ArcGIS tools and geoprocessing functions.
import os     # Enables interaction with the operating system (e.g., file paths).

# Set the workspace: Replace with the actual path to your workspace
workspace = r"E:/wi/mhp-inspected-duplicate-cleaned-grouped-overlayed-thresholded-geotiff"  
arcpy.env.workspace = workspace

# Output directory for shapefiles
output_directory = r"E:/wi/shapefile"  # Replace with the desired path to save the output shapefiles

# Create the output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Create an empty list to store the paths of intermediate shapefiles
intermediate_shapefiles = []

# Iterate through GeoTIFF files in the workspace
for geotiff_file in arcpy.ListFiles("*.tif"):
    # Create the output shapefile path for each group
    group_name = os.path.splitext(os.path.basename(geotiff_file))[0]  # Use file name as group name
    group_output_dir = os.path.join(output_directory, f"group_{group_name}")
    group_shapefile_path = os.path.join(group_output_dir, f"group_{group_name}.shp")
    
    # Create the group's output directory if it doesn't exist
    if not os.path.exists(group_output_dir):
        os.makedirs(group_output_dir)
    
    # Import the GeoTIFF into a raster object
    raster = arcpy.sa.Raster(os.path.join(workspace, geotiff_file))
    
    # Convert the raster to a polygon feature class
    arcpy.conversion.RasterToPolygon(raster, group_shapefile_path, "NO_SIMPLIFY", "VALUE")
    
    # Add a field to store the group information
    arcpy.management.AddField(group_shapefile_path, "Group", "TEXT")
    arcpy.management.CalculateField(group_shapefile_path, "Group", f"'{group_name}'", "PYTHON3")
    
    # Append the intermediate shapefile path to the list
    intermediate_shapefiles.append(group_shapefile_path)